In [1]:
print("Jay Ganesh")

Jay Ganesh


### Creating a Question Answer application over Graph Database

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
NEO4J_URI=os.getenv("NEO4J_URI")
NEO4J_USERNAME=os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD=os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE=os.getenv("NEO4J_DATABASE")
AURA_INSTANCEID=os.getenv("AURA_INSTANCEID")
AURA_INSTANCENAME=os.getenv("AURA_INSTANCENAME")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
os.environ["NEO4J_URI"]=NEO4J_URI
os.environ["NEO4J_USERNAME"]=NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"]=NEO4J_PASSWORD

In [3]:
# from langchain_community.graphs import Neo4jGraph

In [3]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database= NEO4J_DATABASE  # or whatever db you want
)


C:\Users\Jay\anaconda3\envs\ch25\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
graph

In [10]:
moviesq = """
LOAD CSV WITH HEADERS FROM 
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/5e3939d10216b7663687217c1646c30eb921d92f/movies/movies_small.csv' as row

MERGE (m:Movie {id: row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imbdRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') |
    MERGE (p:Person {name: trim(director)})
    MERGE (p)-[:Directed]->(m)
)
FOREACH (actor in split(row.actors, '|') |
    MERGE (p:Person {name: trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m)
)
FOREACH (genre in split(row.genres, '|') |
    MERGE (g:Genre {name: trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g)
)
"""


In [11]:
moviesq

"\nLOAD CSV WITH HEADERS FROM \n'https://raw.githubusercontent.com/tomasonjo/blog-datasets/5e3939d10216b7663687217c1646c30eb921d92f/movies/movies_small.csv' as row\n\nMERGE (m:Movie {id: row.movieId})\nSET m.released = date(row.released),\n    m.title = row.title,\n    m.imbdRating = toFloat(row.imdbRating)\nFOREACH (director in split(row.director, '|') |\n    MERGE (p:Person {name: trim(director)})\n    MERGE (p)-[:Directed]->(m)\n)\nFOREACH (actor in split(row.actors, '|') |\n    MERGE (p:Person {name: trim(actor)})\n    MERGE (p)-[:ACTED_IN]->(m)\n)\nFOREACH (genre in split(row.genres, '|') |\n    MERGE (g:Genre {name: trim(genre)})\n    MERGE (m)-[:IN_GENRE]->(g)\n)\n"

In [12]:
graph.query(moviesq)

[]

In [13]:
graph.refresh_schema()

In [14]:
print(graph.schema)

Node properties:
Movie {id: STRING, released: DATE, title: STRING, imbdRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:Directed]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [16]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
groq_api=os.getenv("GROQ_API_KEY")

In [19]:
llm=ChatGroq(api_key=groq_api,model="openai/gpt-oss-20b")

In [20]:
from langchain_neo4j import GraphCypherQAChain

In [22]:
chain=GraphCypherQAChain.from_llm(graph=graph,llm=llm,verbose=True,allow_dangerous_requests=True)

In [23]:
chain

GraphCypherQAChain(verbose=True, graph=<langchain_neo4j.graphs.neo4j_graph.Neo4jGraph object at 0x000001A7D85A3EB0>, cypher_generation_chain=PromptTemplate(input_variables=['examples', 'question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nExamples (optional):\n{examples}\n\nThe question is:\n{question}')
| RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False

In [28]:
response=chain.invoke({"query":"How many movie did Martin Scorsese directed?"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: 'Martin Scorsese'})-[:Directed]->(m:Movie)
RETURN count(m) AS directedCount;
Full Context:
[{'directedCount': 2}]

> Finished chain.


{'query': 'How many movie did Martin Scorsese directed?',
 'result': 'Martin Scorsese directed 2 movies.'}

In [30]:
response=chain.invoke({"query":"Tell me details about movie Tom and Huck who acted in which genre it comes in?"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: "Tom and Huck"})-[:IN_GENRE]->(g:Genre)
OPTIONAL MATCH (p:Person)-[:ACTED_IN]->(m)
RETURN m.id, m.released, m.title, m.imbdRating, g.name AS genre, collect(p.name) AS actors;
Full Context:
[{'m.id': '8', 'm.released': neo4j.time.Date(1995, 12, 22), 'm.title': 'Tom and Huck', 'm.imbdRating': 5.6, 'genre': 'Adventure', 'actors': ['Jonathan Taylor Thomas', 'Brad Renfro', 'Eric Schweig', 'Charles Rocket']}, {'m.id': '8', 'm.released': neo4j.time.Date(1995, 12, 22), 'm.title': 'Tom and Huck', 'm.imbdRating': 5.6, 'genre': 'Children', 'actors': ['Jonathan Taylor Thomas', 'Brad Renfro', 'Eric Schweig', 'Charles Rocket']}]

> Finished chain.


{'query': 'Tell me details about movie Tom and Huck who acted in which genre it comes in?',
 'result': 'Tom and Huck was released on December\u202f22,\u202f1995. It has an IMDb rating of 5.6 and is classified in the Adventure and Children genres. The film stars Jonathan\u202fTaylor\u202fThomas, Brad\u202fRenfro, Eric\u202fSchweig, and Charles\u202fRocket.'}